In [17]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import os

BASE_DIR = "/content/drive/MyDrive"
MODEL_DIR = os.path.join(BASE_DIR, "model")

gate_dir = os.path.join(MODEL_DIR, "ate_gate_teacher_phase1")
ate_dir = os.path.join(MODEL_DIR, "ate_teacher_phase1")


print(f"Gate model: {gate_dir}")
print(f"  Files: {os.listdir(gate_dir)}")
print(f"ATE model: {ate_dir}")
print(f"  Files: {os.listdir(ate_dir)}")

Gate model: /content/drive/MyDrive/model/ate_gate_teacher_phase1
  Files: ['config.json', 'tokenizer_config.json', 'best_train_loss.json', 'training_args.bin', 'tokenizer.json', 'model.safetensors']
ATE model: /content/drive/MyDrive/model/ate_teacher_phase1
  Files: ['checkpoint-342', 'tokenizer.json', 'best_train_loss.json', 'config.json', 'tokenizer_config.json', 'model.safetensors', 'checkpoint-228']


In [19]:
import os
import pickle
import zipfile
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForTokenClassification

GATE_MODEL_PATH = os.path.join(BASE_DIR, "model", "ate_gate_teacher_phase1")
ATE_MODEL_PATH = os.path.join(BASE_DIR, "model", "ate_teacher_phase1")
DATA_ZIP_PATH = "/content/drive/MyDrive/cleaned2_Kindle_Store.parquet.zip"
DATA_PARQUET_DIR = "/content/cleaned2_Kindle_Store.parquet"
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
REPORT_DIR = os.path.join(OUTPUT_DIR, "reports", "ate_1")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")

GATE_THRESHOLD = 0.9
GATE_NEGATIVE_THRESHOLD = 0.1
SPAN_THRESHOLD = 0.85
SAVE_EVERY = 50_000

device = torch.device("cuda")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Device: {device}")
print(f"Base dir: {BASE_DIR}")
print(f"Exists: {os.path.exists(BASE_DIR)}")

Device: cuda
Base dir: /content/drive/MyDrive
Exists: True


In [20]:
with zipfile.ZipFile(DATA_ZIP_PATH, "r") as zf:
        zf.extractall("/content")

df = pd.read_parquet(DATA_PARQUET_DIR)
print(f"Loaded {len(df)} samples")
print(f"Columns: {df.columns.tolist()}")
df.head()

Loaded 9294220 samples
Columns: ['parent_asin', 'review_id', 'sentence_id', 'sentence_text', 'rating']


,parent_asin,review_id,sentence_id,sentence_text,rating
0,B005KGJXAW,2040,1,dragonbreath travels to the sargasso sea in or...,2.0
1,B005KGJXAW,2040,2,not sure how kids feel about this,2.0
2,B08B4ZPQ9W,2250,1,as two opposites attract who are so full of li...,5.0
3,B08B4ZPQ9W,2250,2,youll fall in love with mercy goode a young wo...,5.0
4,B08B4ZPQ9W,2250,3,and for the first [GENERIC_NOUN] in her life h...,5.0


In [21]:
gate_tok = AutoTokenizer.from_pretrained(GATE_MODEL_PATH)
gate_model = AutoModelForSequenceClassification.from_pretrained(GATE_MODEL_PATH).half().to(device).eval()

ate_tok = AutoTokenizer.from_pretrained(ATE_MODEL_PATH, add_prefix_space=True)
ate_model = AutoModelForTokenClassification.from_pretrained(ATE_MODEL_PATH).half().to(device).eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [22]:
@torch.no_grad()
def predict_gate(sentences, tokenizer, model, batch_size=64, checkpoint_path=None):
    if checkpoint_path and os.path.exists(checkpoint_path):
        confidences = np.load(checkpoint_path).tolist()
        print(f"  Resumed gate checkpoint: {len(confidences)}/{len(sentences)}")
    else:
        confidences = []

    start = len(confidences)
    if start >= len(sentences):
        return confidences

    for i in tqdm(range(start, len(sentences), batch_size), desc="Gate", unit="batch"):
        batch = sentences[i : i + batch_size]
        inputs = tokenizer(
            batch, padding=True, truncation=True, max_length=512, return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
        confidences.extend(probs[:, 1].cpu().tolist())

        if checkpoint_path and (len(confidences) - start) % SAVE_EVERY < batch_size:
            np.save(checkpoint_path, np.array(confidences))

    if checkpoint_path:
        np.save(checkpoint_path, np.array(confidences))
    return confidences


@torch.no_grad()
def extract_aspects(sentences, tokenizer, model, batch_size=32, checkpoint_path=None):
    if checkpoint_path and os.path.exists(checkpoint_path):
        with open(checkpoint_path, "rb") as f:
            ckpt = pickle.load(f)
        all_aspects, all_confidences = ckpt["aspects"], ckpt["confidences"]
        print(f"  Resumed ATE checkpoint: {len(all_aspects)}/{len(sentences)}")
    else:
        all_aspects, all_confidences = [], []

    start = len(all_aspects)
    if start >= len(sentences):
        return all_aspects, all_confidences

    for i in tqdm(range(start, len(sentences), batch_size), desc="ATE", unit="batch"):
        batch = sentences[i : i + batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
            return_offsets_mapping=True,
            is_split_into_words=False,
        )
        offset_mapping = enc.pop("offset_mapping")
        enc = {k: v.to(device) for k, v in enc.items()}

        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        preds = np.argmax(probs, axis=-1)
        offsets = offset_mapping.numpy()

        for j in range(len(batch)):
            sentence = batch[j]
            aspects, confs = [], []
            span_start, span_end = None, None
            span_confs = []

            for k in range(preds.shape[1]):
                s, e = int(offsets[j][k][0]), int(offsets[j][k][1])
                if s == 0 and e == 0:
                    if span_start is not None:
                        text = sentence[span_start:span_end].strip()
                        if text:
                            aspects.append(text)
                            confs.append(round(float(np.mean(span_confs)), 4))
                        span_start, span_end, span_confs = None, None, []
                    continue

                label = int(preds[j][k])

                if label == 1:  # B-ASP
                    if span_start is not None:
                        text = sentence[span_start:span_end].strip()
                        if text:
                            aspects.append(text)
                            confs.append(round(float(np.mean(span_confs)), 4))
                    span_start, span_end = s, e
                    span_confs = [probs[j][k][1]]
                elif label == 2 and span_start is not None:  # I-ASP
                    span_end = e
                    span_confs.append(probs[j][k][2])
                else:  # O
                    if span_start is not None:
                        text = sentence[span_start:span_end].strip()
                        if text:
                            aspects.append(text)
                            confs.append(round(float(np.mean(span_confs)), 4))
                        span_start, span_end, span_confs = None, None, []

            if span_start is not None:
                text = sentence[span_start:span_end].strip()
                if text:
                    aspects.append(text)
                    confs.append(round(float(np.mean(span_confs)), 4))

            all_aspects.append(aspects)
            all_confidences.append(confs)

        if checkpoint_path and (len(all_aspects) - start) % SAVE_EVERY < batch_size:
            with open(checkpoint_path, "wb") as f:
                pickle.dump({"aspects": all_aspects, "confidences": all_confidences}, f)

    if checkpoint_path:
        with open(checkpoint_path, "wb") as f:
            pickle.dump({"aspects": all_aspects, "confidences": all_confidences}, f)
    return all_aspects, all_confidences


def is_high_confidence(gate_conf, aspect_confs):
    if gate_conf <= GATE_NEGATIVE_THRESHOLD:
        return True
    if gate_conf >= GATE_THRESHOLD and any(c >= SPAN_THRESHOLD for c in aspect_confs):
        return True
    return False

In [23]:
os.makedirs(REPORT_DIR, exist_ok=True)

sentences = df["sentence_text"].astype(str).tolist()
gate_ckpt = os.path.join(CHECKPOINT_DIR, "gate_confs.npy")
ate_ckpt = os.path.join(CHECKPOINT_DIR, "ate_results.pkl")

# Gate prediction
print(f"Gate inference on {len(sentences)} sentences...")
gate_confs = predict_gate(sentences, gate_tok, gate_model, checkpoint_path=gate_ckpt)
df["gate_confidence"] = gate_confs

# ATE prediction (only where gate >= threshold)
high_gate_mask = df["gate_confidence"] >= GATE_THRESHOLD
high_gate_idx = df.index[high_gate_mask].tolist()
high_gate_sentences = [sentences[i] for i in high_gate_idx]

print(f"ATE inference on {len(high_gate_sentences)} sentences (gate >= {GATE_THRESHOLD})...")
df["aspects"] = [[] for _ in range(len(df))]
df["confidences"] = [[] for _ in range(len(df))]

if high_gate_sentences:
    asp_list, conf_list = extract_aspects(
        high_gate_sentences, ate_tok, ate_model, checkpoint_path=ate_ckpt,
    )
    for idx, asp, conf in zip(high_gate_idx, asp_list, conf_list):
        df.at[idx, "aspects"] = asp
        df.at[idx, "confidences"] = conf

# Split high / low confidence
high_mask = df.apply(
    lambda r: is_high_confidence(r["gate_confidence"], r["confidences"]), axis=1,
)

high_df = df.loc[
    high_mask,
    ["parent_asin", "review_id", "sentence_id", "sentence_text", "rating",
     "gate_confidence", "aspects", "confidences"],
].reset_index(drop=True)

low_df = df.loc[
    ~high_mask,
    ["parent_asin", "review_id", "sentence_id", "sentence_text", "rating", "gate_confidence"],
].reset_index(drop=True)

# Save parquet
high_path = os.path.join(OUTPUT_DIR, "high_confidence_samples_kindle_store.parquet")
low_path = os.path.join(OUTPUT_DIR, "low_confidence_samples_kindle_store.parquet")
high_df.to_parquet(high_path, index=False)
low_df.to_parquet(low_path, index=False)

# Thong ke
n_high = len(high_df)
n_low = len(low_df)
n_neg = int((high_df["gate_confidence"] <= GATE_NEGATIVE_THRESHOLD).sum())
n_pos = int((high_df["gate_confidence"] >= GATE_THRESHOLD).sum())
has_aspect_count = int(high_df["aspects"].apply(len).gt(0).sum())
total_aspects = int(high_df["aspects"].apply(len).sum())
has_aspect_ratio = has_aspect_count / n_high if n_high else 0
avg_aspects = total_aspects / n_high if n_high else 0

report = (
    f"Báo cáo - Kindle Store\n"
    f"Tổng số mẫu: {len(df)}\n"
    f"Mẫu độ tin cậy cao (high confidence): {n_high}\n"
    f"Mẫu độ tin cậy thấp (low confidence): {n_low}\n\n"
    f"Chi tiet mau do tin cậy cao:\n"
    f"  Tín hiệu âm (gate <= {GATE_NEGATIVE_THRESHOLD}): {n_neg}\n"
    f"  Tín hiệu dương (gate >= {GATE_THRESHOLD}): {n_pos}\n\n"
    f"Tỉ lệ câu có aspect: {has_aspect_ratio:.4f} ({has_aspect_count}/{n_high})\n"
    f"Tổng số aspect được trích xuất: {total_aspects}\n"
    f"Trung bình aspect/mau: {avg_aspects:.4f}\n"
)

report_path = os.path.join(REPORT_DIR, "ate_1_report_kindle_store.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print(report)

Gate inference on 9294220 sentences...


Gate:   0%|          | 0/145223 [00:00<?, ?batch/s]

ATE inference on 4385448 sentences (gate >= 0.9)...


ATE:   0%|          | 0/137046 [00:00<?, ?batch/s]

Báo cáo - Kindle Store
Tổng số mẫu: 9294220
Mẫu độ tin cậy cao (high confidence): 4790509
Mẫu độ tin cậy thấp (low confidence): 4503711

Chi tiet mau do tin cậy cao:
  Tín hiệu âm (gate <= 0.1): 3653848
  Tín hiệu dương (gate >= 0.9): 1136661

Tỉ lệ câu có aspect: 0.2373 (1136661/4790509)
Tổng số aspect được trích xuất: 3463245
Trung bình aspect/mau: 0.7229



In [24]:
with_asp = high_df[high_df["aspects"].apply(len) > 0]
print(f"kindle_store | high={len(high_df)}, low={len(low_df)}, with_aspects={len(with_asp)}")
if len(with_asp) > 0:
    row = with_asp.iloc[0]
    print(f"  Example: \"{row['sentence_text'][:90]}\"")
    print(f"    gate={row['gate_confidence']:.4f}  aspects={row['aspects']}  confs={row['confidences']}")


kindle_store | high=4790509, low=4503711, with_aspects=1136661
  Example: "i loved the rags to riches story and the devotion this family had for their dog wallace"
    gate=0.9907  aspects=['the', 'r', 'story', 'devotion', 'family', 'their', 'dog wallace']  confs=[0.4954, 0.8726, 0.4565, 0.5361, 0.6362, 0.3967, 0.7437]
